In [1]:
from itertools import product
import shutil

In [2]:
#arguments
#expect
    #one starting config file for each wd_pt model
model_size = 'llama-1B-20BT'
wd_pt_lst = [0.1, 0.5, 1.0]
sft_dataset = 'simplescaling'

#default hyperparams
default = ('1.0e-5', 16, 0.0) #lr, bs, wd_ft

In [3]:
#hyperparamter values to sweep: learning rate, batch size, weight decay during finetuning
lr_lst = ['1.0e-5', '3.0e-5', '6.0e-4']
bs_lst = [8, 16, 32] #actual batch size = batch_size * 4 GPUs = 32, 64, 128
wd_ft_lst = [0.0, 0.1, 1.0]

In [4]:
#create hyperparam combos
combos = list(product(lr_lst, bs_lst, wd_ft_lst))

#remove default combo from combos
combos = [combo for combo in combos if combo != default]

### checks
assert len(combos) == len(lr_lst) * len(bs_lst) * len(wd_ft_lst) - 1 #check combo length
assert default not in combos #check that default is not in combos

In [5]:
n_newconfig_files_created = 0

#make a copy of config file for metamathqa
print(f"Creating config file for: model {model_size}, sft_dataset {sft_dataset}")

for wd_pt in wd_pt_lst:
    print(f'wd_pretrain {wd_pt}')

    for i, combo in enumerate(combos):
        lr, bs, wd_ft = combo
        print(f'   {i+1}. lr = {lr}, bs = {bs}, wd_ft = {wd_ft}')

        ### copy config file
        og_config_file_path = f'config_hub/custom_configs/sweep_hyperparams/ft_{sft_dataset}/{model_size}-weightdecay{wd_pt}-seed42-{sft_dataset}.yaml'
        new_config_file_path = f'config_hub/custom_configs/sweep_hyperparams/ft_{sft_dataset}/{model_size}-weightdecay{wd_pt}-seed42-{sft_dataset}--sweep-lr{lr}-bs{bs}-wdft{wd_ft}.yaml'
        shutil.copyfile(og_config_file_path, new_config_file_path)

        ### make edits to the new config file
        with open(new_config_file_path, "r", encoding="utf-8") as f:
            text = f.read()

            #changes these variables in config file:
                # output_dir: llamafactory_out/llama-1B-20BT-weightdecay0.1-seed42-simplescaling
                # per_device_train_batch_size: 16
                # learning_rate: 1.0e-5
                # weight_decay: 0.0
                # run_name: llama-1B-20BT-weightdecay0.1-seed42-simplescaling

            #change output_dir
            old_str = f"output_dir: llamafactory_out/{model_size}-weightdecay{wd_pt}-seed42-{sft_dataset}"
            new_str = f"output_dir: /n/netscratch/doshi-velez_lab/Everyone/models/sweep/{model_size}-weightdecay{wd_pt}-seed42-{sft_dataset}--sweep-lr{lr}-bs{bs}-wdft{wd_ft}"
            text = text.replace(old_str, new_str)

            #change per_device_train_batch_size
            old_str = "per_device_train_batch_size: 16"
            new_str = f"per_device_train_batch_size: {bs}"
            text = text.replace(old_str, new_str)

            #change learning_rate
            old_str = "learning_rate: 1.0e-5"
            new_str = f"learning_rate: {lr}"
            text = text.replace(old_str, new_str)

            #change weight_decay
            old_str = "weight_decay: 0.0"
            new_str = f"weight_decay: {wd_ft}"
            text = text.replace(old_str, new_str)

            #change weight_decay
            old_str = f"run_name: {model_size}-weightdecay{wd_pt}-seed42-{sft_dataset}"
            new_str = f"run_name: {model_size}-weightdecay{wd_pt}-seed42-{sft_dataset}--sweep-lr{lr}-bs{bs}-wdft{wd_ft}"
            text = text.replace(old_str, new_str)
            
            #write new config file
            with open(new_config_file_path, "w", encoding="utf-8") as f:
                f.write(text)

        n_newconfig_files_created += 1

print('# new config files created: ', n_newconfig_files_created)
print("Complete!")

Creating config file for: model llama-1B-20BT, sft_dataset simplescaling
wd_pretrain 0.1
   1. lr = 1.0e-5, bs = 8, wd_ft = 0.0
   2. lr = 1.0e-5, bs = 8, wd_ft = 0.1
   3. lr = 1.0e-5, bs = 8, wd_ft = 1.0
   4. lr = 1.0e-5, bs = 16, wd_ft = 0.1
   5. lr = 1.0e-5, bs = 16, wd_ft = 1.0
   6. lr = 1.0e-5, bs = 32, wd_ft = 0.0
   7. lr = 1.0e-5, bs = 32, wd_ft = 0.1
   8. lr = 1.0e-5, bs = 32, wd_ft = 1.0
   9. lr = 3.0e-5, bs = 8, wd_ft = 0.0
   10. lr = 3.0e-5, bs = 8, wd_ft = 0.1
   11. lr = 3.0e-5, bs = 8, wd_ft = 1.0
   12. lr = 3.0e-5, bs = 16, wd_ft = 0.0
   13. lr = 3.0e-5, bs = 16, wd_ft = 0.1
   14. lr = 3.0e-5, bs = 16, wd_ft = 1.0
   15. lr = 3.0e-5, bs = 32, wd_ft = 0.0
   16. lr = 3.0e-5, bs = 32, wd_ft = 0.1
   17. lr = 3.0e-5, bs = 32, wd_ft = 1.0
   18. lr = 6.0e-4, bs = 8, wd_ft = 0.0
   19. lr = 6.0e-4, bs = 8, wd_ft = 0.1
   20. lr = 6.0e-4, bs = 8, wd_ft = 1.0
   21. lr = 6.0e-4, bs = 16, wd_ft = 0.0
   22. lr = 6.0e-4, bs = 16, wd_ft = 0.1
   23. lr = 6.0e-4, bs = 16